In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r "/content/drive/MyDrive/clean_all_with_bouquet_crops" "new_classification_dataset"

In [ ]:
!pip -q install timm==0.9.16 scikit-learn tqdm

In [ ]:
import os, random, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

import timm
from tqdm.auto import tqdm

from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt

In [ ]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
MIN_PER_CLASS = 15

TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

WEIGHT_DECAY = 1e-4
OUT_ROOT = "/content/models_output"
os.makedirs(OUT_ROOT, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# Підготовка та розподіл набору даних

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")

def prepare_dataset(root):
    class_to_files = {}
    for cls in sorted(os.listdir(root)):
        cls_path = os.path.join(root, cls)
        if not os.path.isdir(cls_path):
            continue
        imgs = [
            os.path.join(cls_path, f)
            for f in os.listdir(cls_path)
            if f.lower().endswith(IMG_EXTS)
        ]
        if len(imgs) >= MIN_PER_CLASS:
            class_to_files[cls] = imgs

    classes = sorted(class_to_files.keys())
    label_map = {c: i for i, c in enumerate(classes)}

    by_class = defaultdict(list)
    for cls, imgs in class_to_files.items():
        y = label_map[cls]
        for p in imgs:
            by_class[y].append(p)

    train, val, test = [], [], []
    for y, paths in by_class.items():
        random.shuffle(paths)
        n = len(paths)
        n_tr = int(n * TRAIN_RATIO)
        n_val = int(n * VAL_RATIO)

        train += [(p, y) for p in paths[:n_tr]]
        val   += [(p, y) for p in paths[n_tr:n_tr+n_val]]
        test  += [(p, y) for p in paths[n_tr+n_val:]]

    return train, val, test, label_map

# Формування датасету та аугментація зображень

In [ ]:
class FlowersDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, y = self.items[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), y

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

val_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE*1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

In [ ]:
def build_model(name, num_classes):
    return timm.create_model(
        name,
        pretrained=True,
        num_classes=num_classes
    ).to(device)

def compute_class_weights(items, num_classes):
    counts = np.zeros(num_classes)
    for _, y in items:
        counts[y] += 1
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)

def make_weighted_sampler(items):
    labels = [y for _, y in items]
    class_counts = np.bincount(labels)
    sample_weights = [1.0 / class_counts[y] for y in labels]
    return WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

# Реалізація процесу навчання моделей

In [ ]:
def train_model(model, train_loader, val_loader, lr, out_dir, label_map, class_weights):
    os.makedirs(out_dir, exist_ok=True)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=0.05
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY
    )

    history = {
        "train_acc": [],
        "val_acc": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": []
    }

    best_f1 = 0.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        correct, total = 0, 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

        train_acc = correct / total
        history["train_acc"].append(train_acc)

        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                preds = model(x).argmax(1)
                y_true.extend(y.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        val_acc = (y_true == y_pred).mean()
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )

        history["val_acc"].append(val_acc)
        history["val_precision"].append(precision)
        history["val_recall"].append(recall)
        history["val_f1"].append(f1)

        print(
            f"Epoch {epoch}: "
            f"train_acc={train_acc:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_f1={f1:.4f}"
        )

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), f"{out_dir}/best_model.pth")
            with open(f"{out_dir}/label_map.json", "w") as f:
                json.dump(label_map, f, indent=2)

    return history

# Експерименти

In [ ]:
EXPERIMENTS = [
    ("vit_base_patch16_224", 3e-5),
    ("convnext_tiny",        3e-4),
    ("convnext_small",       2e-4),
]

DATASETS = {
    "dataset_B": "new_classification_dataset",
}

histories = {}

for ds_name, ds_path in DATASETS.items():
    print(f"\n===== DATASET: {ds_name} =====")

    train_items, val_items, test_items, label_map = prepare_dataset(ds_path)
    num_classes = len(label_map)

    class_weights = compute_class_weights(train_items, num_classes)
    sampler = make_weighted_sampler(train_items)

    train_loader = DataLoader(
        FlowersDataset(train_items, train_tfms),
        batch_size=BATCH_SIZE,
        sampler=sampler
    )

    val_loader = DataLoader(
        FlowersDataset(val_items, val_tfms),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    for model_name, lr in EXPERIMENTS:
        print(f"\n--- Model: {model_name} ---")

        model = build_model(model_name, num_classes)
        out_dir = f"{OUT_ROOT}/{model_name}_{ds_name}"

        history = train_model(
            model,
            train_loader,
            val_loader,
            lr,
            out_dir,
            label_map,
            class_weights
        )

        histories[(ds_name, model_name)] = history

In [ ]:
for (dataset, model), h in histories.items():
    plt.figure(figsize=(6,4))
    plt.plot(h["train_acc"], label="Train Acc")
    plt.plot(h["val_acc"], label="Val Acc")
    plt.plot(h["val_f1"], label="Val F1 (macro)")
    plt.title(f"{model} — {dataset}")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(alpha=0.4)
    plt.show()

In [ ]:
import pandas as pd
import numpy as np

rows = []

for (dataset, model), h in histories.items():
    best_idx = int(np.argmax(h["val_f1"]))

    row = {
        "dataset": dataset,
        "model": model,
        "best_epoch": best_idx + 1,
        "val_accuracy": h["val_acc"][best_idx],
        "val_precision_macro": h["val_precision"][best_idx],
        "val_recall_macro": h["val_recall"][best_idx],
        "val_f1_macro": h["val_f1"][best_idx],
    }
    rows.append(row)

df_metrics = pd.DataFrame(rows)

df_metrics = df_metrics.sort_values(
    by="val_f1_macro",
    ascending=False
).reset_index(drop=True)

df_metrics.round(4)
